In [1]:
!pip install -q transformers wordfreq spacy nltk
!python -m spacy download en_core_web_sm -q
!pip install -U accelerate
import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 MB 30.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.1/183.1 kB 13.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 72.5 MB/s eta 0:00:0000:010:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...


True

In [2]:
import sys, shutil, os

PKG_DIR = "/kaggle/working/reverse_dict"
SRC_DIR = "/kaggle/input/datasets/litrntr/reverse-dict-src"

os.makedirs(PKG_DIR, exist_ok=True)

for fname in ["config.py", "model.py", "data_loader.py", "train.py", "inference.py"]:
    shutil.copy(f"{SRC_DIR}/{fname}", f"{PKG_DIR}/{fname}")

with open(f"{PKG_DIR}/__init__.py", "w") as f:
    f.write("")

sys.path.insert(0, "/kaggle/working")

In [3]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


In [7]:
!cd /kaggle/working/reverse_dict && accelerate launch --multi_gpu --num_processes 2 --num_machines 1 --dynamo_backend no --mixed_precision fp16 train.py

[INFO] Config loaded. Device: cuda
[INFO] Config loaded. Device: cuda
[INFO] Starting training on 2 GPUs...
Loading data from data/dictionary_data.json (train)...
Loaded 66372 pairs for 'train'.
Loading data from data/dictionary_data.json (val)...
Loaded 7375 pairs for 'val'.
Loading weights: 100%|█| 199/199 [00:00<00:00, 1992.18it/s, Materializing param=
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored w

In [17]:
import os

pkg = "/kaggle/working/reverse_dict"

fixes = {
    "model.py":       [("from config import", "from reverse_dict.config import")],
    "data_loader.py": [("from .config import", "from reverse_dict.config import")],
    "train.py":       [("from .config import", "from reverse_dict.config import"),
                       ("from .data_loader import", "from reverse_dict.data_loader import"),
                       ("from .model import", "from reverse_dict.model import")],
    "inference.py":   [("from .config import", "from reverse_dict.config import"),
                       ("from .model import", "from reverse_dict.model import")],
}

for fname, replacements in fixes.items():
    path = os.path.join(pkg, fname)
    with open(path, "r") as f:
        content = f.read()
    for old, new in replacements:
        content = content.replace(old, new)
    with open(path, "w") as f:
        f.write(content)
    print(f"Fixed {fname}")

Fixed model.py
Fixed data_loader.py
Fixed train.py
Fixed inference.py


In [21]:
import os
os.chdir("/kaggle/working/reverse_dict")

In [26]:
# ── Build Embedding Index ─────────────────────────────────────────────
import json, os, sys, torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer
from tqdm import tqdm

sys.path.insert(0, "/kaggle/working")
from reverse_dict.config import Config
from reverse_dict.model import RDModel


class WordOnlyDataset(Dataset):
    def __init__(self, tokenizer):
        file_path = os.path.join("data", Config.LOCAL_DATA_FILE)
        with open(file_path, "r", encoding="utf-8") as f:
            raw = json.load(f)
        self.words = [
            str(w).lower() for w, d in raw.items()
            if d and len(str(d).split()) > 3
        ]
        self.tokenizer = tokenizer
        print(f"[INFO] {len(self.words)} words to index.")

    def __len__(self):
        return len(self.words)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.words[idx],
            max_length=Config.MAX_LEN_WORD,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].flatten(),
            "attention_mask": enc["attention_mask"].flatten(),
        }


# Load model
tokenizer = AutoTokenizer.from_pretrained(Config.MODEL_NAME)
dataset = WordOnlyDataset(tokenizer)
loader = DataLoader(dataset, batch_size=256, shuffle=False, num_workers=2)

model = RDModel(Config.MODEL_NAME).to(Config.DEVICE)
state = torch.load(Config.MODEL_SAVE_PATH, map_location=Config.DEVICE)
model.load_state_dict(state)
model.eval()

# Encode all words
all_embeddings = []
with torch.no_grad():
    for batch in tqdm(loader, desc="Encoding words"):
        ids  = batch["input_ids"].to(Config.DEVICE)
        mask = batch["attention_mask"].to(Config.DEVICE)
        emb  = model.encode_word(ids, mask)
        all_embeddings.append(emb.cpu())

embeddings = F.normalize(torch.cat(all_embeddings, dim=0), p=2, dim=-1)

torch.save(
    {"words": dataset.words, "embeddings": embeddings},
    Config.EMBEDDINGS_SAVE_PATH,
)
print(f"[INFO] Index saved → {Config.EMBEDDINGS_SAVE_PATH}  |  shape: {embeddings.shape}")

[INFO] 73747 words to index.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Encoding words: 100%|██████████| 289/289 [01:12<00:00,  3.97it/s]


[INFO] Index saved → ./checkpoints/word_embeddings_index.pt  |  shape: torch.Size([73747, 256])
